[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhiyunli/cpsc5610-labs/blob/main/week6/week6_lab_solution.ipynb)

# Week 6 Lab: Deep Computer Vision with CNNs

This lab walks through the Week 6 deck. Each section embeds a slide,
then hands you a partially-filled code cell. Replace every
`[YOUR CODE]` with your own implementation; the scaffolding (imports,
names, signatures) is fixed.

By the end you will have:

1. Applied a hand-crafted convolution to real images and verified the
   output-shape formula.
2. Compared max-pool, avg-pool, **depth-wise** max-pool, and **global
   average pool** (3 different ways).
3. Built **LeNet-5**, **TinyAlexNet**, a deeper **CNN** with the
   `DefaultConv2d` partial, and **MiniVGG** on FashionMNIST.
4. Implemented a **`ResidualUnit`** (the ResNet building block) and
   used it to build a small ResNet for FashionMNIST.

The lab is sized for a Colab CPU. With a GPU, every training cell
finishes in seconds.


![title slide](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-01.jpg)

**Setup.** Run this once. If you're missing a package, install it now
(`pip install torch torchvision scikit-learn`).


In [ ]:
import sys
assert sys.version_info >= (3, 10), "Python 3.10+ required"

import math
import time
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("torch version:", torch.__version__)
print("device:", device)


---
# Part 0 --- Why CNNs at all?  The MLP parameter problem

![why is computer vision hard?](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-02.jpg)

The slide claims an MLP on a 224x224 RGB image is parameter-doomed.
Let's verify by counting the very first layer.

**Your task:** for an MLP first layer that flattens a `3x224x224` image
into a `1000`-unit hidden layer, count the parameters; then count the
parameters of a `Conv2d(3, 64, kernel_size=3, padding=1)` operating on
the same input. Print both and the ratio.


In [ ]:
H, W, Cin = 224, 224, 3
mlp_params = (Cin * H * W) * 1000 + 1000     # W * x + b

conv = nn.Conv2d(Cin, 64, kernel_size=3, padding=1)
cnn_params = sum(p.numel() for p in conv.parameters())

print(f"MLP first layer:  {mlp_params:>14,d} params")
print(f"CNN first layer:  {cnn_params:>14,d} params")
print(f"ratio (MLP/CNN):  {mlp_params / cnn_params:>14,.0f}x")


![convolutional layer vs MLP](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-15.jpg)

**What you should see.** ~150M MLP parameters versus a few thousand
for the conv layer --- a ratio of tens of thousands. That single
calculation is the entire reason CNNs exist.


---
# Part 1 --- Convolution on real images

![Section 1: the convolutional layer](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-11.jpg)

We'll start by loading two sample images from scikit-learn
(`china.jpg` and `flower.jpg`), permuting them into PyTorch's
channels-first layout, and pushing them through a `Conv2d`.


In [ ]:
from sklearn.datasets import load_sample_images
import torchvision.transforms.v2 as T

sample_images = np.stack(load_sample_images()["images"])           # (2, 427, 640, 3)
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255.0
images = sample_images.permute(0, 3, 1, 2)                         # (2, 3, 427, 640)

# Center-crop to 70x120 for a small, fast demo.
images = T.CenterCrop((70, 120))(images)
print("images:", tuple(images.shape))

def show(t, **kw):
    plt.imshow(t.permute(1, 2, 0).clamp(0, 1) if t.dim() == 3 else t, **kw)
    plt.axis('off')

plt.figure(figsize=(7, 3))
plt.subplot(1, 2, 1); show(images[0])
plt.subplot(1, 2, 2); show(images[1])
plt.show()


![connections and zero padding](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-17.jpg)

## 1.1 The Conv2d output-shape formula

For input size $H \times W$, kernel $K$, padding $P$, stride $S$:

$$H_\text{out} = \left\lfloor \frac{H + 2P - K}{S} \right\rfloor + 1$$

**Your task:** write `out_size(h, k, p, s)` and confirm it against
`nn.Conv2d` for a few configurations (no padding, "same" padding, large
kernel, stride 2).


In [ ]:
def out_size(h, k, p, s):
    return (h + 2 * p - k) // s + 1

configs = [(70, 7, 0, 1), (70, 7, 3, 1), (70, 5, 0, 1), (70, 7, 3, 2)]
for h, k, p, s in configs:
    conv = nn.Conv2d(3, 1, kernel_size=k, padding=p, stride=s)
    actual = conv(images).shape[-2]
    pred = out_size(h, k, p, s)
    flag = "OK" if pred == actual else "MISMATCH"
    print(f"H={h}, K={k}, P={p}, S={s}  ->  pred={pred}, actual={actual}   {flag}")


![reducing dimensionality with stride](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-18.jpg)

## 1.1b Stride as learned downsampling

Stride 2 is the modern alternative to MaxPool for shrinking spatial
size --- but unlike pooling, it learns a downsampling filter.

**Your task:** apply the same `Conv2d(3, 1, kernel_size=7, padding=3)`
with strides 1, 2, and 4 to the same image, print the output shapes,
and visualize.


In [ ]:
torch.manual_seed(0)
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for i, s in enumerate([1, 2, 4]):
    conv = nn.Conv2d(3, 1, kernel_size=7, padding=3, stride=s)
    out = conv(images[:1])
    ax[i].imshow(out[0, 0].detach(), cmap='gray'); ax[i].axis('off')
    ax[i].set_title(f"stride={s}: {tuple(out.shape)}")
plt.tight_layout(); plt.show()


**What you should see.** Each doubling of stride halves both H and W:
the same 70x120 input becomes ~70x120, ~35x60, ~18x30. With stride
the network learns the downsampling itself --- which is why many
modern architectures replace MaxPool with strided convs entirely.


![stacking multiple feature maps](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-22.jpg)

## 1.2 Two hand-crafted line filters

Before backprop chooses filters for us, let's build two by hand: a
**vertical line** detector and a **horizontal line** detector. We'll
push the sample images through both with `F.conv2d`.

**Your task:** populate two filters of shape `(2, 3, 7, 7)` ---
`filters[0]` should have a vertical white line in column 3, and
`filters[1]` should have a horizontal white line in row 3. Apply with
`padding="same"` so the output keeps the input spatial size.


In [ ]:
filters = torch.zeros([2, 3, 7, 7])
filters[0, :, :, 3] = 1.0
filters[1, :, 3, :] = 1.0

biases = torch.zeros([2])
fmaps = F.conv2d(images, filters, biases, stride=1, padding="same")
print("feature maps:", tuple(fmaps.shape))

plt.figure(figsize=(8, 4))
for image_idx in (0, 1):
    for fmap_idx in (0, 1):
        plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
        plt.imshow(fmaps[image_idx, fmap_idx], cmap='gray')
        plt.axis('off')
plt.suptitle("vertical filter (cols 0,2)  |  horizontal filter (cols 1,3)", y=1.02)
plt.tight_layout()
plt.show()


**What you should see.** The vertical-line filter responds along
vertical edges of the image; the horizontal-line filter responds along
horizontal edges. Backprop learns filters of this character automatically
when you train a CNN on real data.


![multi-channel input -> multi-channel output](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-23.jpg)

## 1.3 The multi-channel parameter formula

For a Conv2d with kernel $K \times K$, $C_\text{in}$ input channels
and $C_\text{out}$ output channels:

$$\#\text{params} = K \cdot K \cdot C_\text{in} \cdot C_\text{out} \;+\; C_\text{out}$$

**Your task:** for each row in the table below, compute the parameter
count by hand using the formula, then verify against
`sum(p.numel() for p in conv.parameters())`.


In [ ]:
rows = [
    (3,    64, 3),
    (64,  128, 3),
    (256, 512, 3),
    (3,    64, 7),
]
print(f"{'Cin':>4} {'Cout':>5} {'K':>3} {'formula':>12} {'actual':>12}")
print("-" * 42)
for Cin, Cout, K in rows:
    pred = K * K * Cin * Cout + Cout
    conv = nn.Conv2d(Cin, Cout, K)
    actual = sum(p.numel() for p in conv.parameters())
    flag = "OK" if pred == actual else "MISMATCH"
    print(f"{Cin:>4} {Cout:>5} {K:>3} {pred:>12,d} {actual:>12,d}   {flag}")


**Key takeaway.** Conv parameter count does NOT depend on the input
spatial size --- only on the kernel and channel counts. Compare to an
MLP layer, where the number of parameters scales with H*W. This is
the second reason CNNs scale to large images.


---
# Part 2 --- Pooling Layers

![Section 2: pooling layers](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-27.jpg)

Three flavors:

1. **Spatial max / avg pool** (`nn.MaxPool2d`, `nn.AvgPool2d`) ---
   shrinks each H, W by the kernel.
2. **Depth-wise max pool** --- pools across *channels* rather than
   space. Used in Xception-style architectures.
3. **Global average pool** (`nn.AdaptiveAvgPool2d(1)`) --- collapses
   each channel to a single number, replacing the giant FC head.


## 2.1 Spatial max-pool and avg-pool

**Your task:** apply `nn.MaxPool2d(2)` and `nn.AvgPool2d(2)` to the
sample images and show the results side by side. Note that with default
`stride=kernel_size`, both pools halve H and W.


In [ ]:
max_pool = nn.MaxPool2d(kernel_size=2)
avg_pool = nn.AvgPool2d(kernel_size=2)

out_max = max_pool(images)
out_avg = avg_pool(images)
print("input :", tuple(images.shape))
print("max   :", tuple(out_max.shape))
print("avg   :", tuple(out_avg.shape))

fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(images[0].permute(1, 2, 0)); ax[0].set_title("input"); ax[0].axis('off')
ax[1].imshow(out_max[0].permute(1, 2, 0)); ax[1].set_title("max pool"); ax[1].axis('off')
ax[2].imshow(out_avg[0].permute(1, 2, 0)); ax[2].set_title("avg pool"); ax[2].axis('off')
plt.tight_layout(); plt.show()


![why pooling helps: translation invariance](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-28.jpg)

## 2.1b Pooling = a free dose of translation invariance

The slide claims that shifting the input by one pixel barely changes
the max-pool output: the maximum within each window is robust to small
shifts. Let's prove it.

**Your task:** take an image, shift it by 1 pixel to the right, run
both the original and the shifted image through `nn.MaxPool2d(2)`, and
compute the average absolute difference between (a) the two raw inputs
and (b) the two pooled outputs. The pooled outputs should be much
closer.


In [ ]:
img = images[:1]
shifted = torch.roll(img, shifts=1, dims=-1)

pool = nn.MaxPool2d(2)
raw_diff = (img - shifted).abs().mean().item()
pool_diff = (pool(img) - pool(shifted)).abs().mean().item()

print(f"avg |raw  - shifted_raw |:  {raw_diff:.4f}")
print(f"avg |pool - shifted_pool|:  {pool_diff:.4f}")
print(f"pooling reduced disagreement by {raw_diff/max(pool_diff,1e-9):.1f}x")


## 2.2 Depth-wise max-pool

PyTorch doesn't ship a depth-wise pool out of the box. The standard
trick is to reshape the tensor to merge spatial dims, then use
`F.max_pool1d` along the channel axis.

**Your task:** complete the `forward` method. Output should have
`channels // kernel_size` channels, same H and W.


In [ ]:
class DepthMaxPool2d(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding

    def forward(self, x):
        B, C, H, W = x.shape
        z = x.view(B, C, H * W)
        z = z.permute(0, 2, 1)
        z = F.max_pool1d(z, self.kernel_size, self.stride, self.padding)
        z = z.permute(0, 2, 1)
        return z.view(B, -1, H, W)

x = torch.randn(2, 8, 16, 16)
print("input  :", tuple(x.shape))
print("dwpool :", tuple(DepthMaxPool2d(kernel_size=2)(x).shape))


![a typical CNN architecture](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-33.jpg)

## 2.3 Three ways to implement Global Average Pool

GAP is the standard replacement for the giant FC head; it cuts
parameters dramatically with no real accuracy loss. There are three
common ways to compute it.

**Your task:** complete the three forms and verify they give the same
output shape `(2, 3, 1, 1)` for our `(2, 3, 70, 120)` input.


In [ ]:
gap_v1 = nn.AvgPool2d(kernel_size=(70, 120))
gap_v2 = nn.AdaptiveAvgPool2d(output_size=1)

out1 = gap_v1(images)
out2 = gap_v2(images)
out3 = images.mean(dim=(2, 3), keepdim=True)

print("v1 (AvgPool2d 70x120)         :", tuple(out1.shape))
print("v2 (AdaptiveAvgPool2d(1))     :", tuple(out2.shape))
print("v3 (.mean(dim=(2,3)))         :", tuple(out3.shape))
print("all equal? ", torch.allclose(out1, out2) and torch.allclose(out1, out3))


**Why this matters.** The AdaptiveAvgPool2d form is the recommended
one --- it's robust to changes in input size and reads more clearly.
The slide's "modern variation: replace FC with GAP" boils down to one
line of code.


---
## Data: FashionMNIST

We use the standard FashionMNIST split. To keep the CPU runtime short,
we take a small subset for training. On a GPU you can scale the
`Subset(...)` size up.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_full = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tfm)
test_set   = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tfm)

N_TRAIN, N_VALID, N_TEST = 6000, 1000, 2000
g = torch.Generator().manual_seed(42)
train_set, valid_set = random_split(
    Subset(train_full, range(N_TRAIN + N_VALID)),
    [N_TRAIN, N_VALID], generator=g)
test_set = Subset(test_set, range(N_TEST))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
valid_loader = DataLoader(valid_set, batch_size=256, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=0)

CLASSES = ["T-shirt","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Boot"]
print(f"train={len(train_set)}, valid={len(valid_set)}, test={len(test_set)}")


### Training helpers

A simple train-one-epoch loop and an accuracy helper.


In [ ]:
def train_one_epoch(model, opt, loader, loss_fn=None):
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * xb.size(0); n += xb.size(0)
    return total / n

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, n = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(dim=1) == yb).sum().item()
        n += yb.size(0)
    return correct / n

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def quick_train(model, epochs=2, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []
    for ep in range(epochs):
        loss = train_one_epoch(model, opt, train_loader)
        val  = accuracy(model, valid_loader)
        history.append((loss, val))
        print(f"  epoch {ep+1}/{epochs}: loss={loss:.4f}  val_acc={val:.4f}")
    return history


---
![data augmentation](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-40.jpg)

## Data augmentation

The slide notes that AlexNet's image augmentations cut ImageNet top-5
error by roughly 3 points. Below we apply a typical training-time
transform stack to a handful of FashionMNIST samples and visualize the
result. The training augmentations *should* preserve label semantics
(a horizontally flipped t-shirt is still a t-shirt) while increasing
input variety.

**Your task:** complete the `T.Compose([...])` list with three
augmentations: a random horizontal flip, a small random rotation, and
a random crop back to 28x28 from a slightly padded image.


In [ ]:
import torchvision.transforms as TT

augment = TT.Compose([
    TT.RandomHorizontalFlip(p=0.5),
    TT.RandomRotation(degrees=10),
    TT.RandomCrop(28, padding=4),
    TT.ToTensor(),
])

raw_set = datasets.FashionMNIST(root="./data", train=True, download=False, transform=None)
fig, ax = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    img, lbl = raw_set[i]
    ax[0, i].imshow(img, cmap='gray'); ax[0, i].axis('off')
    ax[0, i].set_title(CLASSES[lbl], fontsize=9)
    ax[1, i].imshow(augment(img).squeeze(0), cmap='gray'); ax[1, i].axis('off')
ax[0, 0].set_ylabel("original",  rotation=0, labelpad=40); ax[0, 0].axis('on'); ax[0, 0].set_xticks([]); ax[0, 0].set_yticks([])
ax[1, 0].set_ylabel("augmented", rotation=0, labelpad=40); ax[1, 0].axis('on'); ax[1, 0].set_xticks([]); ax[1, 0].set_yticks([])
plt.tight_layout(); plt.show()


**Note.** The bottom row shows realistic perturbations: a flipped
sneaker is still a sneaker, a rotated bag is still a bag. We aren't
plugging this into the training loop in this lab, but in a real
project you would set `transform=augment` on the *training* dataset
only --- never on validation or test.


---
# Part 3 --- LeNet-5 (1998)

![roadmap to classic CNN architectures](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-36.jpg)

![LeNet-5 architecture and PyTorch code](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-37.jpg)

LeNet-5 is the original conv-net that worked: two conv+pool blocks then
two FC layers, trained on 28x28 grayscale digits.

**Your task:** complete the second conv+activation pair and train for
two epochs.


In [ ]:
class LeNet5(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),  nn.Tanh(),
            nn.AvgPool2d(2, 2),
            nn.Conv2d(6, 16, kernel_size=5),            nn.Tanh(),
            nn.AvgPool2d(2, 2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
            nn.Linear(120, 84),         nn.Tanh(),
            nn.Linear(84, n_classes),
        )
    def forward(self, x):
        return self.head(self.body(x))

torch.manual_seed(0)
lenet = LeNet5()
print("LeNet-5 params:", n_params(lenet))
hist_lenet = quick_train(lenet, epochs=2)
print("test_acc:", accuracy(lenet, test_loader))


---
# Part 3.5 --- Tiny AlexNet (2012)

![AlexNet architecture and PyTorch code](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-38.jpg)

AlexNet was the 2012 ImageNet winner. It is essentially LeNet scaled
up with three new ideas:

- **ReLU** activations instead of Tanh.
- **Dropout** in the dense head as regularization.
- **Local Response Normalization** between early conv layers (later
  abandoned in favor of BatchNorm).

Real AlexNet expected a 227x227 input. Our FashionMNIST images are
28x28, so we use a "Tiny AlexNet" with the same *shape* of network but
much smaller layers.

**Your task:** complete the second `Conv2d -> ReLU -> MaxPool` group.


In [ ]:
class TinyAlexNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 96, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(96, 96, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(96, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 3 * 3, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128),         nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.head(self.body(x))

torch.manual_seed(0)
alexnet = TinyAlexNet()
print("TinyAlexNet params:", n_params(alexnet))
hist_alex = quick_train(alexnet, epochs=2)
print("test_acc:", accuracy(alexnet, test_loader))


![why AlexNet beat LeNet](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-39.jpg)

**What you should see.** TinyAlexNet should outperform LeNet despite
having a similar receptive-field structure: ReLU + Dropout do most of
the work. Real AlexNet's other ingredients --- LRN, large batch on 2
GPUs, ImageNet-scale data augmentation --- mattered too, but on a tiny
problem ReLU and Dropout dominate.


---
# Part 4 --- A deeper CNN with `DefaultConv2d`

A handy idiom: use `functools.partial` to fix the most common Conv2d
arguments once and reuse them throughout the model definition.


In [ ]:
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

# Smoke test
print(DefaultConv2d(1, 8))
print(DefaultConv2d(1, 8, kernel_size=7))   # one-off override is fine


## 4.1 A deeper FashionMNIST CNN

The stack: a wide 7x7 first conv to capture large patterns, then 3x3
convs in pairs with channel doubling and `MaxPool2d(2)` between stages,
then a small dense head with dropout.

**Your task:** complete the missing `DefaultConv2d` calls so that
channels go `1 -> 64 -> 128 -> 256` with a pool after each stage.


In [ ]:
torch.manual_seed(42)
deep_cnn = nn.Sequential(
    DefaultConv2d(1, 64, kernel_size=7), nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(64, 128), nn.BatchNorm2d(128), nn.ReLU(),
    DefaultConv2d(128, 128), nn.BatchNorm2d(128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(128, 256), nn.BatchNorm2d(256), nn.ReLU(),
    DefaultConv2d(256, 256), nn.BatchNorm2d(256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(256 * 3 * 3, 128), nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 10),
)
print("Deep-CNN params:", n_params(deep_cnn))
hist_deep = quick_train(deep_cnn, epochs=2)
print("test_acc:", accuracy(deep_cnn, test_loader))


**What you should see.** ~1.6M parameters and ~85-88% validation
accuracy after two epochs on our 6K subset. The 7x7 first conv is the
distinctive thing here: it captures large patterns early so the rest of
the network can specialize. With more data (the full 55K-image
training set) and more epochs (20+), this stack reaches ~92-93%.


---
# Part 5 --- MiniVGG  (and why $3 \times 3$ stacks)

![VGG-16 architecture](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-41.jpg)

![why VGG was special](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-42.jpg)

![VGG point 1: 3x3 stacks beat 5x5](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-43.jpg)

VGG's central bet was that two stacked $3 \times 3$ convs see the same
$5 \times 5$ region as one $5 \times 5$ conv, but use **fewer**
parameters and apply **more** nonlinearities.

**Your task:** verify the parameter math for `Cin = Cout = C = 64`.
Predict each count from the formula $K^2 \cdot C^2$, then compare to
PyTorch.


In [ ]:
C = 64
one_5x5 = nn.Conv2d(C, C, kernel_size=5, padding=2, bias=False)
two_3x3 = nn.Sequential(
    nn.Conv2d(C, C, kernel_size=3, padding=1, bias=False),
    nn.Conv2d(C, C, kernel_size=3, padding=1, bias=False),
)

pred_5x5 = 5 * 5 * C * C
pred_3x3 = 2 * 3 * 3 * C * C

actual_5x5 = sum(p.numel() for p in one_5x5.parameters())
actual_3x3 = sum(p.numel() for p in two_3x3.parameters())

print(f"one 5x5 conv  : pred={pred_5x5:>7,d}  actual={actual_5x5:>7,d}")
print(f"two 3x3 convs : pred={pred_3x3:>7,d}  actual={actual_3x3:>7,d}")
print(f"savings of stacking 3x3 over 5x5: {100*(1 - actual_3x3/actual_5x5):.1f}%")


**What you should see.** $25 C^2$ vs $18 C^2$ --- a 28% savings,
plus a second ReLU between the two convs (= more representational
power). That's the entire VGG intuition.

![VGG-style classifier in PyTorch](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-48.jpg)

The slide's MiniVGG is the VGG recipe in miniature: each block is

$$\text{block}(c_\text{in}, c_\text{out}) = \big[\,\text{Conv}\,3{\times}3 + \text{BN} + \text{ReLU}\,\big]^{\times 2} \to \text{MaxPool}\,2{\times}2.$$

We stack two blocks: 1->32 then 32->64. After two pools the spatial size
goes 28 -> 14 -> 7.

**Your task:** complete the second (Conv + BN + ReLU) inside `block(...)`
and add the second block call.


In [ ]:
class MiniVGG(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.MaxPool2d(2, 2),
            )
        self.body = nn.Sequential(
            block(1,   32),
            block(32,  64),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.head(self.body(x))

torch.manual_seed(0)
vgg = MiniVGG()
print("MiniVGG params:", n_params(vgg))
hist_vgg = quick_train(vgg, epochs=2)
print("test_acc:", accuracy(vgg, test_loader))


## 5.1 Where do MiniVGG's parameters live?

**Your task:** count parameters in `vgg.body` (the conv stack) versus
`vgg.head` (the dense classifier). The slide's "FC tail problem" should
show up in miniature.


In [ ]:
body_params = sum(p.numel() for p in vgg.body.parameters())
head_params = sum(p.numel() for p in vgg.head.parameters())
total = body_params + head_params
print(f"conv body : {body_params:>8,d}  ({100*body_params/total:.1f}%)")
print(f"FC head   : {head_params:>8,d}  ({100*head_params/total:.1f}%)")
print(f"total     : {total:>8,d}")


---
# Part 6 --- The `ResidualUnit` (preview of next week)

![VGG point 3: depth wins](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-45.jpg)

![why VGG-19 lost to VGG-16: the depth ceiling](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/vgg_degradation.png)

Plain deep stacks degrade past ~16-20 layers (slide left over from your
deck). The fix is the **residual block**:

$$\text{out} = F(x) + x$$

If a layer should do nothing, it just learns $F(x) \approx 0$. Below
is the standard `ResidualUnit` you'll see in any ResNet implementation.

**Your task:** complete `forward()` so the output is
`ReLU(main(x) + skip(x))`.


In [ ]:
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        DConv = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main = nn.Sequential(
            DConv(in_channels, out_channels, stride=stride),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            DConv(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
        )
        if stride > 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        return F.relu(self.main(x) + self.skip(x))


class TinyResNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(),
        )
        self.stage1 = nn.Sequential(ResidualUnit(32, 32), ResidualUnit(32, 32))
        self.stage2 = nn.Sequential(ResidualUnit(32, 64, stride=2),
                                    ResidualUnit(64, 64))
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x)
        x = self.gap(x).flatten(1)
        return self.head(x)


torch.manual_seed(0)
resnet = TinyResNet()
print("TinyResNet params:", n_params(resnet))
hist_resnet = quick_train(resnet, epochs=2)
print("test_acc:", accuracy(resnet, test_loader))


**Note.** A full ResNet-34 stacks 16 residual units following
`[64]*3 + [128]*4 + [256]*6 + [512]*3`. We use a shrunken 4-unit
version because FashionMNIST is too small to benefit from depth and we
want CPU-friendly runtimes.


## 6.1 Comparison: every model so far

**Your task:** print a small comparison table.


In [ ]:
models = [
    ("LeNet-5",      lenet,     hist_lenet),
    ("Deep-CNN",     deep_cnn,  hist_deep),
    ("MiniVGG",      vgg,       hist_vgg),
    ("TinyResNet",   resnet,    hist_resnet),
]

print(f"{'model':<14} {'#params':>10} {'val_acc':>10} {'test_acc':>10}")
print("-" * 48)
for name, m, hist in models:
    val = hist[-1][1]
    test = accuracy(m, test_loader)
    print(f"{name:<14} {n_params(m):>10,d} {val:>10.4f} {test:>10.4f}")


---
![what you should be able to do now](https://raw.githubusercontent.com/zhiyunli/cpsc5610-labs/main/week6/slides/slide-51.jpg)

## Wrap-up

You now have hands-on experience with every Week 6 building block:

- Verified the conv-output formula and applied **hand-crafted
  vertical/horizontal line filters** to real images.
- Compared **max / avg / depth-wise max / global average** pooling
  --- including three equivalent GAP implementations.
- Built and trained five models on FashionMNIST: **LeNet-5**,
  **TinyAlexNet**, a deeper **CNN** with `DefaultConv2d`, **MiniVGG**
  (the slide's recipe), and a **TinyResNet** built from a
  `ResidualUnit`.

For the discussion this week: pick the model whose behavior surprised
you most, post your numbers, and explain *why* you think it behaved
that way.
